### Vista Dinámica (Dynamic View)
Este cuaderno muestra cómo implementar la seguridad a nivel de datos utilizando vistas dinámicas.

La implementación se centra en dos funcionalidades principales:

- **Seguridad a nivel de fila:** permite restringir los registros que puede visualizar cada usuario, según las reglas de acceso definidas.

- **Enmascaramiento a nivel de columna:** permite controlar cómo se presentan los valores confidenciales, mostrando la información completa o una versión enmascarada en función de los permisos del usuario.

Para demostrar su funcionamiento, se utilizará un conjunto de datos de **Clientes** sencillo.

#### Paso 1: Crear el schema "data_protection" en el catalog "demo"

In [0]:
USE CATALOG demo;
CREATE SCHEMA IF NOT EXISTS data_protection;

In [0]:
USE SCHEMA data_protection;

#### Paso 2: Crear la tabla "customers_dynamic"
Se crea una tabla que almacena información de clientes procedentes de distintos países.

In [0]:
CREATE OR REPLACE TABLE customers_dynamic (
    customer_id INT,
    name STRING,
    country STRING,
    email STRING,
    balance DECIMAL(10,2),
    credit_card STRING
);

#### Paso 3: Insertar datos de muestra
El conjunto de datos contiene registros correspondientes a los países de Perú y Estados Unidos.

Esto permite comprobar el funcionamiento de la seguridad a nivel de fila, verificando que cada usuario solo pueda visualizar los datos correspondientes a su región.

In [0]:
INSERT INTO customers_dynamic VALUES
(2001, 'Carlos', 'Perú', 'carlos@email.com', 8500.00, '4532123456789012'),
(2002, 'Ana', 'Perú', 'ana@email.com', 6200.00, '5214789632147852'),
(2003, 'John', 'USA', 'john@email.com', 9300.00, '6011122233334444'),
(2004, 'Isabel', 'USA', 'isabel@email.com', 9900.00, '6011122233335555'),
(2005, 'Luci', 'Mexico', 'maria@email.com',  4100.00, '3782822463100050');

In [0]:
SELECT * FROM customers_dynamic;

#### Paso 4: Crear la Vista Dinámica (Dynamic View)
La vista dinámica aplica tanto la seguridad a nivel de fila como el enmascaramiento a nivel de columna.

**Seguridad a nivel de fila**

- **Grupo Perú:** devuelve únicamente las filas en las que el país es 'Perú'.
- **Grupo USA:** devuelve únicamente las filas en las que el país es 'USA'.
- **Grupo Admin:** permite visualizar todos los registros.

**Enmascaramiento a nivel de columna**

- **Grupo Admin:** puede consultar las direcciones de correo electrónico y los números de tarjetas de crédito completos.

- **Otros usuarios:** visualizan las direcciones de correo electrónico y los números de tarjetas de crédito enmascarados.

**Función principal: is_account_group_member()**

Esta función comprueba si el usuario actual pertenece a un grupo específico.

Ejemplos:

- is_account_group_member('peru-sg')
- is_account_group_member('usa-sg')
- is_account_group_member('admin-sg')

La función devuelve **TRUE** o **FALSE** según si el usuario pertenece al grupo indicado.

In [0]:
SELECT is_account_group_member('admin-sg'), is_account_group_member('peru-sg')

In [0]:
CREATE OR REPLACE VIEW vw_customers_dynamic
AS
    SELECT customer_id,
           name,
           country,
           CASE 
            WHEN is_account_group_member('admin-sg') THEN email
            ELSE CONCAT(SUBSTR(email, 1, 1), '***@', SPLIT(email, '@')[1])
           END AS email,
           balance,
           CASE
            WHEN is_account_group_member('admin-sg') THEN credit_card
            ELSE CONCAT(REPEAT('*', LENGTH(credit_card) - 4), RIGHT(credit_card, 4))
           END AS credit_card
    FROM customers_dynamic
    WHERE is_account_group_member('admin-sg')
    OR (is_account_group_member('peru-sg') AND country = 'Perú')
    OR (is_account_group_member('usa-sg') AND country = 'USA');

#### Paso 5: Consultando la Vista Dinámica (Dynamic View)

In [0]:
SELECT * FROM vw_customers_dynamic;